# 05 · Polymorphism, duck typing and protocols

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 50 minutes live | [04 · Encapsulation and abstraction](04_encapsulation_and_abstraction.ipynb) | [06 · Code safari](06_code_safari_reading_real_code.ipynb) |

The retention team wants one table that compares every candidate model: their current rule of thumb,
a logistic regression, a random forest, and whatever someone tries next week. If the comparison code
needs an `if isinstance(...)` branch per model type, it breaks every time a new model shows up.

This notebook shows how one function can work with many kinds of objects (polymorphism), why Python
only cares about the methods an object has (duck typing), and how to write type hints that describe
"anything with a `predict` method" (protocols).

> **Learning goals.** By the end you can:
>
> - explain polymorphism with an example from your own model code
> - write one evaluation function that works for your own classes and for scikit-learn models
> - describe the shape of an object with a `typing.Protocol` and say what `@runtime_checkable` does and does not check
> - choose between an abstract base class and a protocol for a given situation and justify it

**Before you start:** notebooks 01 to 04.

> **How to work on the exercises**
>
> - Every exercise is done at the computer, **alone or in pairs**. In pairs, one person types (the driver) and the other reviews and thinks ahead (the navigator); swap at every exercise. Online, the driver shares their screen, or you both use VS Code Live Share.
> - Levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for experienced programmers).
> - Each exercise ends with a **Discuss** question. Talk it through with your partner or the group. Working alone? Write a short answer in a markdown cell.
> - Check cells fail until the exercise is solved; that is expected. Hints and solutions are in the collapsed blocks.
> - **Check your understanding** questions: predict the output first, then run the cell below the question.

**In this notebook**

0. Setup
1. Polymorphism: one function, many models
2. Duck typing: Python only checks what you call
3. Type hints meet duck typing: protocols
4. Mixins: small classes that add one capability

---

## 0. Setup

We load and clean the churn data, split it once, and re-create compact versions of the classes from the
earlier notebooks: `ConfusionMatrix` (notebook 01), the rule baselines (notebook 02) and `RetentionCost`
(notebook 03). Run the cells and move on; nothing here is new.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH).set_index("customerID")
# TotalCharges is text and blank for customers in their first month (tenure 0): they owe nothing yet
customers["TotalCharges"] = pd.to_numeric(customers["TotalCharges"], errors="coerce").fillna(0.0)
customers["Churn"] = (customers["Churn"] == "Yes").astype(int)

X = customers.drop(columns=["Churn"])
y = customers["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train: {len(X_train)} customers, test: {len(X_test)} customers, churn rate {y.mean():.1%}")

In [ ]:
class ConfusionMatrix:
    """Counts of hits and misses of a binary classifier (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t, p = np.asarray(y_true), np.asarray(y_pred)
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if p + r else 0.0

    def __repr__(self) -> str:
        return f"ConfusionMatrix(tn={self.tn}, fp={self.fp}, fn={self.fn}, tp={self.tp})"


class RetentionCost:
    """Money lost to churn when the retention team acts on the model's flags.

    Every flagged customer gets an offer (offer_cost). A churner who gets the offer stays with
    probability acceptance_rate. Every churner who leaves costs customer_value.
    """

    def __init__(
        self, offer_cost: float = 50, customer_value: float = 600, acceptance_rate: float = 0.3
    ):
        self.offer_cost = offer_cost
        self.customer_value = customer_value
        self.acceptance_rate = acceptance_rate

    def __call__(self, y_true, y_pred) -> float:
        cm = ConfusionMatrix.from_predictions(y_true, y_pred)
        offers = (cm.tp + cm.fp) * self.offer_cost
        lost_despite_offer = cm.tp * (1 - self.acceptance_rate) * self.customer_value
        lost_unflagged = cm.fn * self.customer_value
        return offers + lost_despite_offer + lost_unflagged


class BaselineModel:
    """Base class of our hand-written rule models (notebook 02)."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "BaselineModel":
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        raise NotImplementedError

    def evaluate(self, X: pd.DataFrame, y: pd.Series) -> ConfusionMatrix:
        return ConfusionMatrix.from_predictions(y, self.predict(X))


class MajorityClassModel(BaselineModel):
    """Always predicts the most frequent class seen in training (here: nobody churns)."""

    def fit(self, X, y):
        self.majority_ = int(y.mode()[0])
        return self

    def predict(self, X):
        return np.full(len(X), self.majority_)


class ContractRuleModel(BaselineModel):
    """The retention team's rule of thumb: flag every month-to-month customer."""

    def predict(self, X):
        return (X["Contract"] == "Month-to-month").astype(int).to_numpy()


class TenureRuleModel(BaselineModel):
    """Flag customers who joined less than max_tenure months ago."""

    def __init__(self, max_tenure: int = 12):
        self.max_tenure = max_tenure

    def predict(self, X):
        return (X["tenure"] < self.max_tenure).astype(int).to_numpy()


cost = RetentionCost()
print("Setup done.")

---

## 1. Polymorphism: one function, many models

*Polymorphism* (Greek for "many shapes") means: code written against an interface works with every
object that provides that interface. You have used it all along: `len()` works on lists, strings,
DataFrames and your own classes, because each of them provides `__len__`.

For us the interface is "has a `fit(X, y)` and a `predict(X)` method". The function below never asks
*what* model it got. It just calls the two methods.

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test) -> ConfusionMatrix:
    """Fit a model on training data and return its confusion matrix on test data."""
    model.fit(X_train, y_train)
    return ConfusionMatrix.from_predictions(y_test, model.predict(X_test))


for model in [MajorityClassModel(), ContractRuleModel(), TenureRuleModel(max_tenure=12)]:
    cm = evaluate(model, X_train, y_train, X_test, y_test)
    print(f"{type(model).__name__:20s} recall={cm.recall:.2f} precision={cm.precision:.2f}")

So far all three models share a base class. The interesting part: scikit-learn models do **not**
inherit from our `BaselineModel`, yet the same function works with them.

scikit-learn models need numbers, so we wrap each estimator in a small scikit-learn `Pipeline` that
one-hot encodes the categorical columns first. You will meet `Pipeline` again in notebook 06; for now
treat `make_sklearn_model` as "give me a model that accepts our raw customer table".

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CATEGORICAL = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
    "OnlineSecurity",
    "TechSupport",
]


def make_sklearn_model(estimator):
    """Return a scikit-learn pipeline: encode and scale the columns, then fit `estimator`."""
    preprocess = make_column_transformer(
        (StandardScaler(), NUMERIC),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    )
    return make_pipeline(preprocess, estimator)


models = {
    "contract rule": ContractRuleModel(),
    "logistic regression": make_sklearn_model(LogisticRegression(max_iter=1000)),
    "random forest": make_sklearn_model(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42)
    ),
}

for name, model in models.items():
    cm = evaluate(model, X_train, y_train, X_test, y_test)
    print(f"{name:20s} recall={cm.recall:.2f} precision={cm.precision:.2f} f1={cm.f1:.2f}")

`evaluate` has no idea that `ContractRuleModel` is a hand-written rule and the others are scikit-learn
pipelines. Adding a new model next week requires **zero changes** to `evaluate`. That is the practical
value of polymorphism: the code that *uses* models is decoupled from the code that *implements* them.

Notice also what the numbers say: the contract rule catches most churners but flags many loyal customers
too; the learned models are more precise. Which one is "best" depends on what an offer costs, which is
exactly what `RetentionCost` measures.

### Exercise 5.1
> **Core** · for everyone

Write `leaderboard(models, X_train, y_train, X_test, y_test, cost)` that returns a DataFrame with one row
per model and the columns `precision`, `recall`, `f1` and `cost` (money lost, from `cost(y_test, y_pred)`),
sorted by `cost`, lowest first. The index should be the model names.

Then add a fourth model, `make_sklearn_model(GradientBoostingClassifier(random_state=42))`, to the
dictionary **without changing your function**.

**Discuss.** The scikit-learn models need numbers, the rule models need the raw table, yet `leaderboard` treats them all the same. Where does the knowledge "this model needs encoded columns" live, and what would go wrong if you moved it into `leaderboard` instead?

In [ ]:
# Exercise 5.1: your code here

from sklearn.ensemble import GradientBoostingClassifier


def leaderboard(models: dict, X_train, y_train, X_test, y_test, cost) -> pd.DataFrame:
    raise NotImplementedError("Your code here")


all_models = dict(models)  # add the gradient boosting model here

board = leaderboard(all_models, X_train, y_train, X_test, y_test, cost)
board

In [ ]:
# Check cell for exercise 5.1
assert list(board.columns) == ["precision", "recall", "f1", "cost"], board.columns
assert len(board) == 4, "Did you add the gradient boosting model?"
assert board["cost"].is_monotonic_increasing, "Sort by cost, lowest first"
assert board.loc["contract rule", "recall"] > 0.8
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Loop over `models.items()`. For each model call `fit`, then `predict`, build a `ConfusionMatrix`, and
collect a dict like `{"precision": ..., "recall": ..., "f1": ..., "cost": ...}` in a list. Finally
`pd.DataFrame(rows, index=names).sort_values("cost")`.
</details>

<details><summary><b>Show solution</b></summary>

```python
def leaderboard(models: dict, X_train, y_train, X_test, y_test, cost) -> pd.DataFrame:
    rows = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cm = ConfusionMatrix.from_predictions(y_test, y_pred)
        rows[name] = {
            "precision": cm.precision,
            "recall": cm.recall,
            "f1": cm.f1,
            "cost": cost(y_test, y_pred),
        }
    return pd.DataFrame.from_dict(rows, orient="index").sort_values("cost")


all_models = dict(models)
all_models["gradient boosting"] = make_sklearn_model(GradientBoostingClassifier(random_state=42))
```

The function never mentions a model type. Gradient boosting works because it has `fit` and `predict`.
</details>

---

## 2. Duck typing: Python only checks what you call

> "If it walks like a duck and quacks like a duck, it is a duck."

Python does not check the *type* of an object before calling a method on it. It simply looks the
method up when the line runs. If the method is there, the call works. If not, you get an
`AttributeError` at that moment. This is called **duck typing**.

That is why `evaluate` accepted scikit-learn pipelines: nobody checked their class, only their methods.
It also means mistakes surface late, at the line that makes the call:

In [ ]:
class ChurnScoresFromCRM:
    """Pre-computed churn scores exported from a CRM system: there is nothing to fit."""

    def __init__(self, scores: pd.Series):
        self.scores = scores

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (self.scores.loc[X.index] > 0.5).astype(int).to_numpy()


crm = ChurnScoresFromCRM(pd.Series(0.2, index=X.index))
try:
    evaluate(crm, X_train, y_train, X_test, y_test)
except AttributeError as exc:
    print(f"AttributeError: {exc}")

> **Check your understanding**
>
> Suppose `evaluate` did **not** call `model.fit(...)` and only called `model.predict(X_test)`.
> Would `evaluate(crm, ...)` work?
>
> <details><summary><b>Show answer</b></summary>
>
> Yes. Duck typing only requires the methods that are actually called. The CRM object has `predict`,
> so a predict-only `evaluate` works. The required "interface" is defined by the code that uses the
> object, not by a class declaration. This is flexible, and it is also why the interface is easy to get
> wrong without documentation or type hints.
> </details>

In [ ]:
# Predict the output first, then run this cell
def evaluate_predict_only(model, X_test, y_test) -> ConfusionMatrix:
    return ConfusionMatrix.from_predictions(y_test, model.predict(X_test))


print(evaluate_predict_only(crm, X_test, y_test))

### scikit-learn is built on conventions, not on inheritance checks

Most of scikit-learn works by convention: `fit` returns `self`, learned attributes end in an
underscore (`coef_`, `classes_`), and `predict` takes the same columns as `fit`. Many functions only rely
on those conventions.

Some do more, though. `cross_val_score` and `GridSearchCV` make a fresh, unfitted copy of your model for
every fold (they *clone* it). Cloning needs the method `get_params`, which our rule models do not have:

In [ ]:
from sklearn.model_selection import cross_val_score

try:
    cross_val_score(TenureRuleModel(max_tenure=12), X_train, y_train, cv=5, scoring="recall")
except TypeError as exc:
    print(f"TypeError: {exc}")

The message is honest: our object "does not seem to be a scikit-learn estimator". Duck typing still
applies, the required duck is just bigger than `fit` + `predict`. Inheriting from
`sklearn.base.BaseEstimator` adds `get_params` for free; notebook X2 shows how to build fully compatible
estimators.

## 3. Type hints meet duck typing: protocols

How should we annotate the `model` parameter of `evaluate`? A first idea is our base class:

```python
def evaluate(model: BaselineModel, X_train, y_train, X_test, y_test) -> ConfusionMatrix: ...
```

At runtime nothing changes (type hints are not enforced), but a type checker such as `mypy` would now
reject every scikit-learn model:

```text
error: Argument 1 to "evaluate" has incompatible type "Pipeline"; expected "BaselineModel"
```

The hint describes the wrong thing. We do not need "a subclass of `BaselineModel`". We need "anything
with `fit` and `predict`". The two kinds of compatibility have names:

| Kind | Meaning | Example |
|---|---|---|
| nominal (by inheritance) | the class declares its parent | `ContractRuleModel(BaselineModel)` |
| structural (by shape) | the object has the right methods | a scikit-learn `Pipeline` |

A `typing.Protocol` describes a structural type. Classes do **not** inherit from it; they match it if
they have the listed methods.

In [ ]:
from typing import Protocol, runtime_checkable


@runtime_checkable
class Predictor(Protocol):
    """Anything that can be trained and then label customers."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> object: ...

    def predict(self, X: pd.DataFrame) -> np.ndarray: ...


def evaluate(model: Predictor, X_train, y_train, X_test, y_test) -> ConfusionMatrix:
    model.fit(X_train, y_train)
    return ConfusionMatrix.from_predictions(y_test, model.predict(X_test))


for name, model in models.items():
    print(f"{name:20s} is a Predictor: {isinstance(model, Predictor)}")
print(f"{'CRM scores':20s} is a Predictor: {isinstance(crm, Predictor)}")

Neither `ContractRuleModel` nor the scikit-learn pipelines mention `Predictor` anywhere, yet they match
it. The CRM export does not, because it has no `fit`.

`@runtime_checkable` is what makes `isinstance` work with a protocol. It is a shallow check: it only
looks at whether **methods with those names exist**. It does not check the arguments or return types:

In [ ]:
class Misleading:
    def fit(self):  # wrong signature: takes no data at all
        return self

    def predict(self):
        return "not an array"


print("isinstance(Misleading(), Predictor):", isinstance(Misleading(), Predictor))

The full signature check is the job of a static type checker. Running `mypy` on a file that passes
`Misleading()` to `evaluate` reports the mismatch before the code ever runs. That is the main value of
protocols: precise type hints for duck-typed code, checked by your editor and by mypy.

### When to use an ABC, when a Protocol

| Situation | Use |
|---|---|
| You own the class family and want to **share code** (e.g. a concrete `evaluate` or `fit_transform` in the base class) and force subclasses to implement certain methods | abstract base class (notebook 04) |
| You want to **describe what you need** from objects you do not control (scikit-learn, colleagues' code, test doubles) | `Protocol` |
| You need to check at runtime that an object can be used | prefer calling it and handling the error; `isinstance` with a runtime-checkable protocol only checks method names |

Both can coexist: our `BaselineModel` hierarchy shares `evaluate`, and every function that *uses*
models accepts a `Predictor`. The reference package does the same: see `ProbabilisticClassifier` in
`src/churn/model.py` and `CustomerLoader` in `src/churn/data.py`.

### Exercise 5.2
> **Stretch** · if you finish early

1. Annotate your `leaderboard` function from exercise 5.1: `models` is a `dict[str, Predictor]`. For
   `cost`, write a second protocol `CostFunction` with a single method
   `__call__(self, y_true: pd.Series, y_pred: np.ndarray) -> float` (notebook 03: `__call__` is what
   makes `cost(...)` work). Our `RetentionCost` matches it by shape.
2. Write a class `AlwaysFlagModel` that flags every customer (recall 1, the most expensive campaign)
   **without** inheriting from anything, confirm with `isinstance` that it is a `Predictor`, and add it
   to the leaderboard.

If you have time, save your function and models in a file, e.g. `playground/compare.py`, and run
`uv run mypy playground/compare.py` in the terminal. Then pass a `Misleading()` object to your function and
see what mypy says.

**Discuss.** `AlwaysFlagModel` counts as a `Predictor` without inheriting from anything. For the models your team writes, would you rather give them a common base class or only a protocol? What does `isinstance(obj, Predictor)` really prove, and what does it not?

In [ ]:
# Exercise 5.2: your code here


class AlwaysFlagModel: ...

In [ ]:
# Check cell for exercise 5.2
assert isinstance(AlwaysFlagModel(), Predictor)
cm = evaluate(AlwaysFlagModel(), X_train, y_train, X_test, y_test)
assert cm.recall == 1.0 and cm.tn == 0
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class CostFunction(Protocol):
    def __call__(self, y_true: pd.Series, y_pred: np.ndarray) -> float: ...


def leaderboard(
    models: dict[str, Predictor],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    cost: CostFunction,
) -> pd.DataFrame:
    ...  # body unchanged


class AlwaysFlagModel:
    """Send an offer to everyone: maximum recall, maximum campaign cost."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "AlwaysFlagModel":
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return np.ones(len(X), dtype=int)
```

A protocol with only `__call__` describes "something you can call like this function". Notebook 07 shows the shorter spelling for exactly this case, `Callable[[pd.Series, np.ndarray], float]`.

`AlwaysFlagModel` matches `Predictor` purely by its shape. Compare its cost with the do-nothing
`MajorityClassModel` in the leaderboard: neither extreme is the cheapest.
</details>

---

## 4. Mixins: small classes that add one capability

You will see one more pattern constantly in library code: **mixins**. A mixin is a small class that is
not meant to be used on its own. It adds one capability to any class that lists it as an extra parent.

scikit-learn uses this everywhere. Every classifier inherits `ClassifierMixin`, which provides a
`score` method (accuracy) implemented once in terms of `self.predict`:

In [ ]:
from sklearn.base import ClassifierMixin

print("LogisticRegression parents:", [cls.__name__ for cls in LogisticRegression.__mro__[1:5]])
print("score comes from ClassifierMixin:", LogisticRegression.score is ClassifierMixin.score)
print("same for RandomForestClassifier:", RandomForestClassifier.score is ClassifierMixin.score)

`ClassifierMixin.score` works for every classifier because of polymorphism: it calls `self.predict`,
and each classifier brings its own. A class can still override a mixin method: `DummyClassifier`, for
example, defines its own `score`. In notebook 06 you will read this kind of code yourself.

Use mixins sparingly in your own code: one capability per mixin, no `__init__` of its own, and name it
`...Mixin` so readers know it is not a standalone class.

### Exercise 5.3
> **Challenge** · for experienced programmers

The leaderboard compares models at their default decision rule. But logistic regression and random
forests produce probabilities (`predict_proba`), and the retention team can pick any threshold. A model
that loses at threshold 0.5 may win at 0.25.

Write `best_threshold(model, X_val, y_val, cost)` that, **if the model has `predict_proba`**, tries
thresholds from 0.05 to 0.95 in steps of 0.05 on a validation set and returns the threshold with the
lowest cost; for models without `predict_proba` it returns `None`. Then build a leaderboard that uses
each model's best threshold.

Important: do not choose the threshold on the test set; you would be tuning on the data you report.
Split `X_train` once more into a fitting part and a validation part.

**Discuss.** Talk these through with your partner or the group; if you work alone, answer them in a markdown cell:
1. Checking `hasattr(model, "predict_proba")` is duck typing at runtime. What would a second protocol
   `ProbabilisticPredictor` add, and what would it not add?
2. Build `make_sklearn_model(LinearSVC())` (from `sklearn.svm`; a classifier without probabilities).
   `Pipeline` defines a `predict_proba` method in its class, so what do you expect
   `hasattr(pipeline, "predict_proba")` to return? Try it. What does the result teach you about
   writing wrapper classes (like `ChurnModel`) that other code will duck-type?

In [ ]:
# Exercise 5.3: your code here


def best_threshold(model, X_val, y_val, cost) -> float | None:
    raise NotImplementedError("Your code here")

In [ ]:
# Check cell for exercise 5.3
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)
lr = make_sklearn_model(LogisticRegression(max_iter=1000)).fit(X_fit, y_fit)
t = best_threshold(lr, X_val, y_val, cost)
assert t is not None and 0.05 <= t <= 0.95
assert best_threshold(ContractRuleModel().fit(X_fit, y_fit), X_val, y_val, cost) is None
tuned_cost = cost(y_test, (lr.predict_proba(X_test)[:, 1] >= t).astype(int))
print(
    f"best threshold {t:.2f}: test cost {tuned_cost:,.0f} vs {cost(y_test, lr.predict(X_test)):,.0f} at 0.5"
)
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
def best_threshold(model, X_val, y_val, cost) -> float | None:
    if not hasattr(model, "predict_proba"):
        return None
    proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.arange(0.05, 0.96, 0.05)
    costs = [cost(y_val, (proba >= t).astype(int)) for t in thresholds]
    return float(thresholds[int(np.argmin(costs))])
```

1. A protocol `ProbabilisticPredictor` (with `predict_proba`) documents the requirement and lets mypy
   check calls before running anything. It does not add a runtime guarantee: `isinstance` with
   `@runtime_checkable` still only checks that a method with that name exists.
2. It returns `False`, although the method is written in the `Pipeline` class. scikit-learn wraps it
   with a decorator (`available_if`) that makes the attribute disappear when the final step lacks
   `predict_proba`. The library goes out of its way to keep `hasattr` truthful, because so much code
   duck-types on it. A naive wrapper that always defines `predict_proba` and forwards the call would
   make `hasattr` say `True` and then fail when called. Lessons: a wrapper should only advertise the
   methods it can really serve, and when in doubt the robust duck-typing style is "try it and handle
   the error" (EAFP: easier to ask forgiveness than permission).

With the cost model used here, the cheapest threshold for logistic regression is below 0.5:
missing a churner (600) is far more expensive than an unnecessary offer (50).
</details>

---

## Summary

- **Polymorphism**: code that calls methods of an interface works with every object that provides them.
  One `evaluate` or `leaderboard` function serves rule baselines and scikit-learn models alike.
- **Duck typing**: Python checks for a method only when the call happens. The "interface" is whatever the
  calling code uses. Errors appear late, at the call.
- Some library functions need more than `fit`/`predict` (e.g. cloning needs `get_params`). Read the error,
  it usually says what is missing.
- **Protocols** give duck-typed code precise type hints. Classes match them by shape, without inheriting.
  `@runtime_checkable` only checks method names.
- Use an **ABC** to share code inside a family you own; use a **Protocol** to describe what you need from
  objects you do not control.
- **Mixins** add one capability to many classes (`ClassifierMixin.score`).

**Common pitfalls**
- Annotating with your own base class when you mean "anything with these methods".
- Trusting `isinstance(obj, SomeProtocol)` as proof that calls will succeed.
- Writing `if type(model) == ...` branches instead of giving each model the method you need.

## Going further

- Notebook X2 builds estimators that work with `cross_val_score` and `GridSearchCV`.

---

## Sources and further reading

**Sources.** This notebook is based on:

- Python docs: [`typing.Protocol`](https://docs.python.org/3/library/typing.html#typing.Protocol) and the glossary entry [duck typing](https://docs.python.org/3/glossary.html#term-duck-typing).
- [PEP 544, Protocols: Structural subtyping](https://peps.python.org/pep-0544/), the design document with its motivation.
- [mypy: Protocols and structural subtyping](https://mypy.readthedocs.io/en/stable/protocols.html).
- [scikit-learn: Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html), the conventions our duck typing relies on.

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Duck Typing in Python](https://realpython.com/duck-typing-python/) (article, intermediate).
- [Real Python: Python Protocols, Leveraging Structural Subtyping](https://realpython.com/python-protocol/) (article, intermediate).
- Luciano Ramalho, *Fluent Python*, 2nd edition, O'Reilly 2022 ([book site](https://www.fluentpython.com/)), chapter 13 "Interfaces, Protocols, and ABCs" (book, advanced): the four ways of typing in Python and when to use each.

---

[← 04 · Encapsulation and abstraction](04_encapsulation_and_abstraction.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [06 · Code safari →](06_code_safari_reading_real_code.ipynb)